In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip uninstall torchao -y
!pip install detoxify peft pytorch-lightning torchmetrics wandb datasets transformers -U

In [ ]:
%%writefile train_model.py
import os
import re
import csv
import shutil
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from torchmetrics.classification import BinaryAUROC
from transformers import get_linear_schedule_with_warmup
from detoxify import Detoxify
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, TQDMProgressBar
from peft import get_peft_model, LoraConfig, TaskType
from datasets import load_dataset
import datasets
import numpy as np
from sklearn.metrics import f1_score
import copy

# ==========================================
# 0. 全局环境
# ==========================================
os.environ["WANDB_API_KEY"] = "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"
os.environ["WANDB_SILENT"] = "true"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
datasets.disable_progress_bar()

# ==========================================
# 1. Focal Loss（三语毒性比例均为 36.8%）
# ==========================================
class BinaryFocalLoss(nn.Module):
    def __init__(self, gamma=1.0, alpha=0.63):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        focal_loss = alpha_t * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()

# ==========================================
# 2. 单语言 LoRA 模型（一个完整的独立模型）
# ==========================================
class BaseWithHead(nn.Module):
    def __init__(self, encoder, hidden_size):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask, **kwargs):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(pooled))

def build_single_lang_model(base_encoder, hidden_size, lang_name):
    """
    为每种语言构建一个独立的 LoRA 模型。
    三个模型共享同一个 base_encoder（XLM-R 权重冻结），
    但各自有独立的 LoRA 参数和 classifier head。
    """
    base_with_head = BaseWithHead(base_encoder, hidden_size)

    peft_config = LoraConfig(
        task_type=TaskType.FEATURE_EXTRACTION,
        inference_mode=False,
        r=32,
        lora_alpha=64,
        lora_dropout=0.1,
        target_modules=["query", "key", "value", "dense"],
        modules_to_save=["classifier"]
    )

    model = get_peft_model(base_with_head, peft_config)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  {lang_name}: 可训练参数 {trainable:,}")
    return model

# ==========================================
# 3. 三模型容器
# ==========================================
class ThreeModelDetoxify(nn.Module):
    """
    三个物理独立的 LoRA 模型，共享同一个冻结的 XLM-R base encoder。
    训练时按 lang_id 拆分 batch，每个子 batch 走对应的模型。
    不依赖 set_adapter，完全隔离。
    """
    def __init__(self, base_encoder, hidden_size):
        super().__init__()
        self.lang_id_to_name = {0: "en", 1: "de", 2: "fi"}

        print("\n🔧 构建三个独立 LoRA 模型（共享 base encoder）...")
        # 三个模型共享同一个 base_encoder 实例（冻结的 XLM-R）
        self.model_en = build_single_lang_model(base_encoder, hidden_size, "en")
        self.model_de = build_single_lang_model(base_encoder, hidden_size, "de")
        self.model_fi = build_single_lang_model(base_encoder, hidden_size, "fi")

        self.lang_models = {0: self.model_en, 1: self.model_de, 2: self.model_fi}
        self.focal_loss = BinaryFocalLoss()

    def forward_with_lang(self, input_ids, attention_mask, labels=None, lang_ids=None):
        device = input_ids.device
        batch_size = input_ids.shape[0]
        all_logits = torch.zeros(batch_size, device=device)
        total_loss = torch.tensor(0.0, device=device)
        valid_count = 0

        for lang_id, model in self.lang_models.items():
            mask = (lang_ids == lang_id)
            if not mask.any():
                continue

            sub_logits = model(
                input_ids=input_ids[mask],
                attention_mask=attention_mask[mask]
            ).squeeze(-1).float()
            all_logits[mask] = sub_logits

            if labels is not None:
                sub_loss = self.focal_loss(sub_logits, labels[mask].squeeze(-1))
                total_loss = total_loss + sub_loss * mask.sum()
                valid_count += mask.sum()

        if labels is not None and valid_count > 0:
            return {"loss": total_loss / valid_count, "logits": all_logits}
        return {"logits": all_logits}

# ==========================================
# 4. Lightning Module
# ==========================================
class DetoxifyFineTuner(pl.LightningModule):
    def __init__(self, model, lr=2e-5):
        super().__init__()
        self.model = model
        self.lr = lr
        self.target_langs = ["en", "de", "fi"]
        self.lang_idx_map = {"en": 0, "de": 1, "fi": 2}

    def forward(self, **batch):
        return self.model.forward_with_lang(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch.get("labels"),
            lang_ids=batch["lang_ids"]
        )

    def training_step(self, batch, batch_idx):
        out = self(**batch)
        self.log("train_loss", out["loss"], sync_dist=True)
        return out["loss"]

    def on_validation_epoch_start(self):
        self.val_probs = {lang: [] for lang in self.target_langs}
        self.val_targets = {lang: [] for lang in self.target_langs}

    def validation_step(self, batch, batch_idx):
        out = self(**batch)
        probs = torch.sigmoid(out["logits"])
        labels = batch["labels"].squeeze(-1).int()
        lang_ids = batch["lang_ids"]
        self.log("val_loss", out["loss"], sync_dist=True)

        for lang, idx in self.lang_idx_map.items():
            mask = (lang_ids == idx)
            if mask.any():
                self.val_probs[lang].append(probs[mask].detach().cpu())
                self.val_targets[lang].append(labels[mask].detach().cpu())

    def on_validation_epoch_end(self):
        valid_aucs = []
        valid_best_f1s = []
        current_lang_thresholds = {}
        all_global_probs = []
        all_global_targets = []

        for lang in self.target_langs:
            if len(self.val_probs[lang]) == 0:
                continue

            full_probs = torch.cat(self.val_probs[lang])
            full_targets = torch.cat(self.val_targets[lang])

            self.val_probs[lang].clear()
            self.val_targets[lang].clear()

            auc_metric = BinaryAUROC()
            auc_val = auc_metric(full_probs, full_targets.int())
            valid_aucs.append(auc_val)

            all_global_probs.append(full_probs)
            all_global_targets.append(full_targets)

            probs_np = full_probs.numpy()
            targets_np = full_targets.numpy()

            best_f1, best_t = 0.0, 0.5
            for t in np.arange(0.1, 0.92, 0.02):
                f1 = f1_score(targets_np, (probs_np >= t).astype(int), average='macro', zero_division=0)
                if f1 > best_f1:
                    best_f1, best_t = f1, t

            current_lang_thresholds[lang] = float(best_t)
            valid_best_f1s.append(torch.tensor(best_f1))

            self.log(f"val_auc_{lang}", auc_val, sync_dist=False)
            self.log(f"val_opt_t_{lang}", float(best_t), sync_dist=False)
            self.log(f"val_f1_{lang}", float(best_f1), sync_dist=False)
            print(f"  {lang}: AUC={auc_val:.4f}, best_t={best_t:.2f}, F1={best_f1:.4f} (n={len(full_probs)})")

        if valid_aucs:
            macro_auc = torch.stack(valid_aucs).mean()
            min_auc = torch.stack(valid_aucs).min()
            self.log("val_macro_auc", macro_auc, prog_bar=True, sync_dist=False)
            self.log("val_min_auc", min_auc, prog_bar=True, sync_dist=False)

        if valid_best_f1s:
            macro_f1 = torch.stack(valid_best_f1s).mean()
            self.log("val_macro_opt_f1", macro_f1, prog_bar=True, sync_dist=False)

            best_global_t = 0.5
            if all_global_probs:
                gp = torch.cat(all_global_probs).numpy()
                gt = torch.cat(all_global_targets).numpy()
                best_gf1 = 0.0
                for t in np.arange(0.1, 0.92, 0.02):
                    f1 = f1_score(gt, (gp >= t).astype(int), average='macro', zero_division=0)
                    if f1 > best_gf1:
                        best_gf1 = f1
                        best_global_t = t

            os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
            with open("/kaggle/working/checkpoints/threshold.json", "w") as f:
                json.dump({"global": float(best_global_t), "per_lang": current_lang_thresholds}, f, indent=2)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=0.01)
        total_steps = self.trainer.estimated_stepping_batches
        warmup_steps = min(int(total_steps * 0.05), 1500)
        scheduler = get_linear_schedule_with_warmup(
            optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
        )
        return [optimizer], [{"scheduler": scheduler, "interval": "step", "frequency": 1}]

# ==========================================
# 5. 执行入口
# ==========================================
if __name__ == '__main__':

    train_path = "/kaggle/input/datasets/lyushiro/snlp22/translated_cleaned.tsv"
    dev_path = "/kaggle/input/datasets/lyushiro/snlp22/dev_cleaned.tsv"
    ckpt_dir = "/kaggle/working/checkpoints"
    os.makedirs(ckpt_dir, exist_ok=True)

    detox = Detoxify("multilingual", device="cpu")
    base_encoder = detox.model.base_model
    hidden_size = base_encoder.config.hidden_size
    tokenizer = detox.tokenizer
    max_length = 256

    raw_datasets = load_dataset(
        "csv",
        data_files={"train": train_path, "validation": dev_path},
        delimiter="\t",
        quoting=csv.QUOTE_NONE,
        keep_default_na=False
    )

    lang_map = {"en": 0, "de": 1, "fi": 2}

    def preprocess_function(examples):
        cleaned_texts = [
            re.sub(r'http\S+|www.\S+', '', re.sub(r'<[^>]+>', ' ', str(t))).strip()
            for t in examples['text']
        ]
        tokenized = tokenizer(cleaned_texts, add_special_tokens=False)
        batch_input_ids = []
        batch_attention_mask = []
        batch_labels = [[float(l)] for l in examples['label']]
        batch_lang_ids = [lang_map.get(str(l).lower(), 0) for l in examples.get('lang', ['en'] * len(examples['text']))]
        max_valid_len = max_length - 2

        for raw_ids in tokenized["input_ids"]:
            if len(raw_ids) > max_valid_len:
                half = max_valid_len // 2
                raw_ids = raw_ids[:half] + raw_ids[-half:]
            input_ids = [tokenizer.cls_token_id] + raw_ids + [tokenizer.sep_token_id]
            pad_len = max_length - len(input_ids)
            input_ids = input_ids + [tokenizer.pad_token_id] * pad_len
            attention_mask = [1] * (max_length - pad_len) + [0] * pad_len
            batch_input_ids.append(input_ids)
            batch_attention_mask.append(attention_mask)

        return {
            "input_ids": batch_input_ids,
            "attention_mask": batch_attention_mask,
            "labels": batch_labels,
            "lang_ids": batch_lang_ids
        }

    print("开始 Arrow 内存映射与分词...")
    tokenized_datasets = raw_datasets.map(
        preprocess_function,
        batched=True,
        num_proc=2,
        remove_columns=raw_datasets["train"].column_names,
    )
    tokenized_datasets.set_format(type="torch", columns=["input_ids", "attention_mask", "labels", "lang_ids"])

    train_loader = DataLoader(
        tokenized_datasets["train"],
        batch_size=32,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True
    )
    val_loader = DataLoader(
        tokenized_datasets["validation"],
        batch_size=64,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True
    )

    # ---------- 构建三个独立模型 ----------
    trainable_model = ThreeModelDetoxify(base_encoder, hidden_size)
    pl_model = DetoxifyFineTuner(trainable_model, lr=2e-5)

    wandb_logger = WandbLogger(
        project="toxic-comment-classification",
        name="three-model-translated-297k",
        log_model=False
    )

    checkpoint_callback = ModelCheckpoint(
        monitor="val_macro_auc",
        mode="max",
        save_top_k=1,
        dirpath=ckpt_dir,
        filename="best-model-{epoch:02d}"
    )

    early_stop = EarlyStopping(
        monitor="val_macro_auc",
        patience=10,
        mode="max",
        verbose=True
    )

    progress_bar = TQDMProgressBar(refresh_rate=500)

    trainer = pl.Trainer(
        max_epochs=3,
        log_every_n_steps=50,
        val_check_interval=500,
        accelerator="gpu",
        devices=1,
        precision="16-mixed",
        logger=wandb_logger,
        callbacks=[checkpoint_callback, early_stop, progress_bar]
    )

    trainer.fit(pl_model, train_loader, val_loader)

    best_model_path = checkpoint_callback.best_model_path
    final_target_path = os.path.join(ckpt_dir, "result.ckpt")
    threshold_path = os.path.join(ckpt_dir, "threshold.json")

    if best_model_path and os.path.exists(best_model_path):
        shutil.copy(best_model_path, final_target_path)
        print(f"\n✅ 最佳模型已保存: {final_target_path}")
        print(f"🏆 最佳 val_macro_auc: {checkpoint_callback.best_model_score:.4f}")

        # 关键修复：
        # threshold.json 在每次 validation 后都会被覆盖，
        # 如果直接沿用训练结束时磁盘上的 threshold.json，
        # 它可能对应“最后一次验证”的模型，而不是 best checkpoint。
        # 所以这里显式基于 best checkpoint 重新跑一次验证，确保 threshold 与 result.ckpt 配套。
        print("\n🔄 正在基于 best checkpoint 重新计算 threshold.json ...")
        if os.path.exists(threshold_path):
            os.remove(threshold_path)

        trainer.validate(model=pl_model, dataloaders=val_loader, ckpt_path=best_model_path)
    else:
        print("\n⚠️ 未找到 best checkpoint，正在使用当前模型重新计算 threshold.json ...")
        trainer.validate(model=pl_model, dataloaders=val_loader)

    print(f"✅ threshold.json: {threshold_path}")


In [ ]:
!python train_model.py

In [ ]:
import os
import re
import json
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from detoxify import Detoxify
from peft import get_peft_model, LoraConfig, TaskType

# ==========================================
# 1. 模型架构（必须与训练代码完全一致）
# ==========================================
class BaseWithHead(torch.nn.Module):
    def __init__(self, encoder, hidden_size):
        super().__init__()
        self.encoder = encoder
        self.dropout = torch.nn.Dropout(0.3)
        self.classifier = torch.nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask, **kwargs):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(pooled))

def build_single_lang_model(base_encoder, hidden_size):
    base_with_head = BaseWithHead(base_encoder, hidden_size)
    peft_config = LoraConfig(
        task_type=TaskType.FEATURE_EXTRACTION,
        inference_mode=True,
        r=32, lora_alpha=64, lora_dropout=0.1,
        target_modules=["query", "key", "value", "dense"],
        modules_to_save=["classifier"]
    )
    return get_peft_model(base_with_head, peft_config)

class ThreeModelDetoxify(torch.nn.Module):
    def __init__(self, base_encoder, hidden_size):
        super().__init__()
        self.model_en = build_single_lang_model(base_encoder, hidden_size)
        self.model_de = build_single_lang_model(base_encoder, hidden_size)
        self.model_fi = build_single_lang_model(base_encoder, hidden_size)
        self.lang_models = {"en": self.model_en, "de": self.model_de, "fi": self.model_fi}

    def forward_single_lang(self, input_ids, attention_mask, lang_key):
        model = self.lang_models[lang_key]
        return model(input_ids=input_ids, attention_mask=attention_mask).squeeze(-1)

# ==========================================
# 2. 推理专用 Dataset
# ==========================================
class TestDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=256):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.max_valid_len = max_length - 2

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = re.sub(r'http\S+|www.\S+', '',
                      re.sub(r'<[^>]+>', ' ', str(self.texts[idx]))).strip()
        tokenized = self.tokenizer(text, add_special_tokens=False)
        raw_ids = tokenized["input_ids"]
        if len(raw_ids) > self.max_valid_len:
            half = self.max_valid_len // 2
            raw_ids = raw_ids[:half] + raw_ids[-half:]
        input_ids = [self.tokenizer.cls_token_id] + raw_ids + [self.tokenizer.sep_token_id]
        pad_len = self.max_length - len(input_ids)
        input_ids = input_ids + [self.tokenizer.pad_token_id] * pad_len
        attention_mask = [1] * (self.max_length - pad_len) + [0] * pad_len
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long)
        }

# ==========================================
# 3. 分语言推理
# ==========================================
def run_inference(test_path, ckpt_path, threshold_path, output_path, batch_size=128):
    device = torch.device("cuda:0")
    print(f"--- 正在使用设备: {device} ---")

    id_prefix_to_lang = {"eng": "en", "ger": "de", "fin": "fi"}

    if os.path.exists(threshold_path):
        with open(threshold_path, "r") as f:
            threshold_data = json.load(f)
        per_lang = threshold_data["per_lang"]
        global_t = threshold_data["global"]
        print(f"✅ 分语言阈值: {per_lang}")
        print(f"✅ 全局阈值 (fallback): {global_t}")
    else:
        print(f"⚠️  阈值文件不存在: {threshold_path}，使用默认 0.5")
        per_lang = {"en": 0.5, "de": 0.5, "fi": 0.5}
        global_t = 0.5

    df_test = pd.read_csv(test_path, sep='\t', header=0, quoting=3)
    texts = df_test["text"].fillna("").tolist()
    print(f"成功加载测试数据: {len(texts)} 条")

    df_test["_lang_prefix"] = df_test["id"].str.split("_").str[0]
    df_test["_lang_key"] = df_test["_lang_prefix"].map(id_prefix_to_lang)
    print(f"语言分布:\n{df_test['_lang_prefix'].value_counts().to_string()}")

    # ---------- 初始化模型 ----------
    print("\n正在初始化预训练底座...")
    detox = Detoxify("multilingual", device="cpu")
    base_encoder = detox.model.base_model
    tokenizer = detox.tokenizer
    hidden_size = base_encoder.config.hidden_size

    model = ThreeModelDetoxify(base_encoder, hidden_size)
    del detox

    # ---------- 加载权重 ----------
    print(f"正在加载权重文件: {ckpt_path}")
    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = checkpoint["state_dict"]
    new_state_dict = {k[len("model."):]: v for k, v in state_dict.items() if k.startswith("model.")}
    model.load_state_dict(new_state_dict, strict=True)
    model.to(device)
    model.eval()
    print("✅ 三模型已就绪\n")

    # ---------- 分语言推理 ----------
    all_probs = np.zeros(len(df_test))

    for id_prefix, lang_key in id_prefix_to_lang.items():
        mask = (df_test["_lang_prefix"] == id_prefix)
        if not mask.any():
            continue

        indices = mask[mask].index.tolist()
        lang_texts = [texts[i] for i in indices]

        print(f"🔄 使用 model_{lang_key} 推理 {id_prefix} ({len(lang_texts)} 条)...")

        test_ds = TestDataset(lang_texts, tokenizer)
        test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

        lang_probs = []
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
            for batch in tqdm(test_loader, desc=f"  {id_prefix}"):
                logits = model.forward_single_lang(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    lang_key
                )
                probs = torch.sigmoid(logits).cpu().numpy()
                lang_probs.extend(probs.tolist())

        for i, idx in enumerate(indices):
            all_probs[idx] = lang_probs[i]

    df_test["_prob"] = all_probs

    # ---------- 分语言应用阈值 ----------
    df_test["predicted"] = 0
    for id_prefix, lang_key in id_prefix_to_lang.items():
        mask = df_test["_lang_prefix"] == id_prefix
        t = per_lang.get(lang_key, global_t)
        df_test.loc[mask, "predicted"] = (df_test.loc[mask, "_prob"] >= t).astype(int)
        n_total = mask.sum()
        n_toxic = df_test.loc[mask, "predicted"].sum()
        print(f"  {id_prefix} ({lang_key}): 阈值={t:.4f}, 样本数={n_total}, 预测毒性={n_toxic}")

    unknown = df_test["_lang_key"].isna()
    if unknown.any():
        df_test.loc[unknown, "predicted"] = (df_test.loc[unknown, "_prob"] >= global_t).astype(int)
        print(f"  未知语言: 使用全局阈值={global_t:.4f}, 样本数={unknown.sum()}")

    df_out = df_test[["id", "predicted"]]
    df_out.to_csv(output_path, sep="\t", index=False)

    print(f"\n{'='*50}")
    print(f"✅ 预测完成！结果已保存至: {output_path}")
    print(f"\n📊 总体预测分布:")
    print(df_test["predicted"].value_counts().to_string())
    print(f"\n📊 分语言预测分布:")
    for pfx in sorted(id_prefix_to_lang.keys()):
        mask = df_test["_lang_prefix"] == pfx
        if mask.any():
            print(f"  {pfx}: {df_test.loc[mask, 'predicted'].value_counts().to_dict()}")
    print(f"{'='*50}")

# ==========================================
# 4. 执行入口
# ==========================================
if __name__ == "__main__":
    TEST_FILE = "/kaggle/input/datasets/lyushiro/snlp22/test_cleaned.tsv"
    MODEL_FILE = "/kaggle/working/checkpoints/result.ckpt"
    THRESHOLD_FILE = "/kaggle/working/checkpoints/threshold.json"
    OUTPUT_FILE = "submission_results.tsv"

    run_inference(TEST_FILE, MODEL_FILE, THRESHOLD_FILE, OUTPUT_FILE)
